# 年別Nested Walk-Forwardの保存実験

歴史的な実験コードです。現在の実行入口は `../09_confidence_nested.ipynb`。
元Notebookのセル番号は0始まりです。コードの個人フォルダ名は置換しています。独立実行は保証しません。
保存出力は `../../results/imported_20260907/`、監査は `../../docs/CONFIDENCE_AUDIT.md` を参照してください。


## 元のセル index 35


In [ ]:
# ============================================================
# NESTED WALK-FORWARD
# 15分足10年 / Confidence Threshold完全OOS検証
#
# 目的
# ------------------------------------------------------------
# モデルだけでなく、
# 「何%以上のConfidenceで取引するか」
# というThreshold選択も未来データを使わずに行う。
#
# 例:
#
# 2023 Test
# 2016-2021 -> Model Train
# 2022      -> ValidationでThreshold選択
# 2023      -> 選んだThresholdを固定してTest
#
# 2024 Test
# 2016-2022 -> Model Train
# 2023      -> Validation
# 2024      -> Test
#
# ------------------------------------------------------------
# TP/SLなし
# BET sizingなし
# 30分保有
# 次足Open Entry
# Cost = 0.004%
# 重複ポジション禁止
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


# ============================================================
# 1. 設定
# ============================================================

CSV_PATH = (
    Path.cwd()
    / "dukascopy_usdjpy"
    / "usdjpy_15m_2016_2026.csv"
)

HORIZON_BARS = 2

RANDOM_STATE = 42

N_ESTIMATORS = 250

TRADING_COST = 0.00004

# Validationで選ぶ候補
THRESHOLD_CANDIDATES = [
    0.50,
    0.52,
    0.54,
    0.55,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
]

# Validationで最低限必要な取引数
MIN_VALIDATION_TRADES = 100

# 最低学習年数
MIN_TRAIN_YEARS = 3


# ============================================================
# 2. 読み込み
# ============================================================

df = pd.read_csv(
    CSV_PATH,
    index_col=0,
    parse_dates=True,
)

df.index = pd.to_datetime(
    df.index,
    utc=True,
)

df = (
    df
    .sort_index()
    .copy()
)

df.columns = [
    c.lower()
    for c in df.columns
]


print("読み込み完了")
print("総行数:", len(df))
print(
    "期間:",
    df.index.min(),
    "→",
    df.index.max(),
)


# ============================================================
# 3. RSI
# ============================================================

def calculate_rsi(
    close,
    period=14,
):

    delta = close.diff()

    gain = delta.clip(
        lower=0
    )

    loss = -delta.clip(
        upper=0
    )

    avg_gain = (
        gain
        .rolling(period)
        .mean()
    )

    avg_loss = (
        loss
        .rolling(period)
        .mean()
    )

    rs = (
        avg_gain
        /
        avg_loss.replace(
            0,
            np.nan
        )
    )

    return (
        100
        -
        100
        /
        (
            1 + rs
        )
    )


# ============================================================
# 4. 特徴量
# ============================================================

def make_features(
    data
):

    x = data.copy()

    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    for n in [
        1,
        2,
        4,
        8,
        16,
    ]:

        x[
            f"return_{n}"
        ] = (
            x["close"]
            .pct_change(n)
        )

    # --------------------------------------------------------
    # Volatility
    # --------------------------------------------------------

    for n in [
        4,
        8,
        16,
        32,
    ]:

        x[
            f"vol_{n}"
        ] = (
            x["return_1"]
            .rolling(n)
            .std()
        )

    # --------------------------------------------------------
    # MA
    # --------------------------------------------------------

    for period in [
        5,
        10,
        20,
        50,
        100,
    ]:

        ma = (
            x["close"]
            .rolling(period)
            .mean()
        )

        x[
            f"ma{period}_distance"
        ] = (
            x["close"]
            /
            ma
            - 1
        )

        x[
            f"ma{period}_slope"
        ] = (
            ma.pct_change()
        )

    # --------------------------------------------------------
    # Candle
    # --------------------------------------------------------

    candle_range = (
        x["high"]
        -
        x["low"]
    ).replace(
        0,
        np.nan
    )

    x["body"] = (
        x["close"]
        -
        x["open"]
    ) / candle_range

    x["upper_wick"] = (
        x["high"]
        -
        x[
            [
                "open",
                "close",
            ]
        ].max(
            axis=1
        )
    ) / candle_range

    x["lower_wick"] = (
        x[
            [
                "open",
                "close",
            ]
        ].min(
            axis=1
        )
        -
        x["low"]
    ) / candle_range

    x["range_pct"] = (
        x["high"]
        -
        x["low"]
    ) / x["close"]

    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    x["rsi14"] = (
        calculate_rsi(
            x["close"],
            14,
        )
        /
        100
    )

    # --------------------------------------------------------
    # ATR
    # --------------------------------------------------------

    prev_close = (
        x["close"]
        .shift(1)
    )

    true_range = pd.concat(
        [
            x["high"]
            -
            x["low"],

            (
                x["high"]
                -
                prev_close
            ).abs(),

            (
                x["low"]
                -
                prev_close
            ).abs(),
        ],
        axis=1,
    ).max(
        axis=1
    )

    x["atr14"] = (
        true_range
        .rolling(14)
        .mean()
        /
        x["close"]
    )

    # --------------------------------------------------------
    # 高値・安値距離
    # --------------------------------------------------------

    high16 = (
        x["high"]
        .rolling(16)
        .max()
    )

    low16 = (
        x["low"]
        .rolling(16)
        .min()
    )

    x["distance_high_16"] = (
        high16
        -
        x["close"]
    ) / x["close"]

    x["distance_low_16"] = (
        x["close"]
        -
        low16
    ) / x["close"]

    # --------------------------------------------------------
    # 時刻
    # --------------------------------------------------------

    hour = (
        x.index.hour
        +
        x.index.minute
        / 60
    )

    x["hour_sin"] = np.sin(
        2
        *
        np.pi
        *
        hour
        /
        24
    )

    x["hour_cos"] = np.cos(
        2
        *
        np.pi
        *
        hour
        /
        24
    )

    x["weekday"] = (
        x.index.dayofweek
        /
        4
    )

    return x


data = make_features(
    df
)


# ============================================================
# 5. Entry / Exit
#
# Signal:
# 足tのCloseで確定
#
# Entry:
# 次足Open
#
# Exit:
# 30分後Close
# ============================================================

data["entry_price"] = (
    data["open"]
    .shift(-1)
)

data["exit_price"] = (
    data["close"]
    .shift(
        -HORIZON_BARS
    )
)

data["future_return"] = (
    data["exit_price"]
    /
    data["entry_price"]
    - 1
)

data["target"] = (
    data["future_return"]
    > 0
).astype(int)


# ============================================================
# 6. 特徴量
# ============================================================

FEATURES = [

    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",

    "ma10_distance",
    "ma10_slope",

    "ma20_distance",
    "ma20_slope",

    "ma50_distance",
    "ma50_slope",

    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",
    "weekday",
]


data = (
    data
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
    .dropna(
        subset=
            FEATURES
            +
            [
                "entry_price",
                "exit_price",
                "future_return",
                "target",
            ]
    )
    .copy()
)


print()
print(
    "使用可能データ:",
    len(data)
)


# ============================================================
# 7. Model作成
# ============================================================

def build_model():

    return RandomForestClassifier(

        n_estimators=
            N_ESTIMATORS,

        max_depth=
            8,

        min_samples_leaf=
            30,

        max_features=
            "sqrt",

        class_weight=
            "balanced",

        random_state=
            RANDOM_STATE,

        n_jobs=
            -1,
    )


# ============================================================
# 8. Prediction Frame
# ============================================================

def predict_frame(
    model,
    frame,
):

    probability = (
        model.predict_proba(
            frame[
                FEATURES
            ]
        )[:, 1]
    )

    result = pd.DataFrame(
        index=
            frame.index
    )

    result[
        "p_up"
    ] = probability

    result[
        "p_down"
    ] = (
        1
        -
        probability
    )

    result[
        "confidence"
    ] = np.maximum(
        result[
            "p_up"
        ],
        result[
            "p_down"
        ]
    )

    result[
        "prediction"
    ] = np.where(
        result[
            "p_up"
        ]
        >=
        0.5,
        1,
        -1,
    )

    result[
        "side"
    ] = np.where(
        result[
            "prediction"
        ]
        ==
        1,
        "BUY",
        "SELL",
    )

    result[
        "future_return"
    ] = (
        frame[
            "future_return"
        ].values
    )

    result[
        "gross_return"
    ] = (
        result[
            "future_return"
        ]
        *
        result[
            "prediction"
        ]
    )

    result[
        "correct"
    ] = (
        result[
            "gross_return"
        ]
        > 0
    )

    return result


# ============================================================
# 9. 重複取引禁止
# ============================================================

def non_overlapping(
    frame,
):

    if frame.empty:
        return frame.copy()

    frame = (
        frame
        .sort_index()
    )

    selected_indices = []

    next_allowed = None

    hold_time = pd.Timedelta(
        minutes=30
    )

    for timestamp in (
        frame.index
    ):

        if (
            next_allowed
            is not None
            and
            timestamp
            <
            next_allowed
        ):

            continue

        selected_indices.append(
            timestamp
        )

        next_allowed = (
            timestamp
            +
            hold_time
        )

    return (
        frame.loc[
            selected_indices
        ]
        .copy()
    )


# ============================================================
# 10. Stats
# ============================================================

def calculate_stats(
    returns
):

    r = np.asarray(
        returns,
        dtype=float
    )

    if len(
        r
    ) == 0:

        return {
            "trades":
                0,

            "accuracy":
                np.nan,

            "avg_return":
                np.nan,

            "pf":
                np.nan,

            "max_dd":
                np.nan,

            "growth":
                np.nan,
        }

    gains = (
        r[
            r > 0
        ].sum()
    )

    losses = (
        -r[
            r < 0
        ].sum()
    )

    if losses > 0:

        pf = (
            gains
            /
            losses
        )

    elif gains > 0:

        pf = np.inf

    else:

        pf = np.nan

    equity = np.r_[
        1,
        np.cumprod(
            1 + r
        )
    ]

    peak = (
        np.maximum.accumulate(
            equity
        )
    )

    drawdown = (
        equity
        /
        peak
        - 1
    )

    return {
        "trades":
            len(r),

        "accuracy":
            (
                r > 0
            ).mean(),

        "avg_return":
            np.mean(
                r
            ),

        "pf":
            pf,

        "max_dd":
            np.min(
                drawdown
            ),

        "growth":
            equity[-1]
            - 1,
    }


# ============================================================
# 11. ValidationでThreshold選択
#
# 注意:
# Profit Factorだけ最大化すると
# サンプル数が少ない高Thresholdを
# 選びやすい。
#
# そこで、
#
# score =
# 平均Return × sqrt(Trades)
#
# を使う。
# ============================================================

def choose_threshold(
    validation_predictions
):

    rows = []

    best_threshold = None

    best_score = -np.inf

    for threshold in (
        THRESHOLD_CANDIDATES
    ):

        candidate = (
            validation_predictions.loc[
                validation_predictions[
                    "confidence"
                ]
                >=
                threshold
            ]
        )

        candidate = (
            non_overlapping(
                candidate
            )
        )

        if (
            len(
                candidate
            )
            <
            MIN_VALIDATION_TRADES
        ):

            continue

        net_return = (
            candidate[
                "gross_return"
            ]
            -
            TRADING_COST
        )

        stats = calculate_stats(
            net_return
        )

        score = (
            stats[
                "avg_return"
            ]
            *
            np.sqrt(
                stats[
                    "trades"
                ]
            )
        )

        rows.append(
            {
                "threshold":
                    threshold,

                "trades":
                    stats[
                        "trades"
                    ],

                "accuracy":
                    stats[
                        "accuracy"
                    ],

                "avg_return":
                    stats[
                        "avg_return"
                    ],

                "pf":
                    stats[
                        "pf"
                    ],

                "score":
                    score,
            }
        )

        if (
            score
            >
            best_score
        ):

            best_score = score

            best_threshold = (
                threshold
            )

    return (
        best_threshold,
        pd.DataFrame(
            rows
        )
    )


# ============================================================
# 12. Nested Walk-Forward
# ============================================================

all_years = sorted(
    data.index.year.unique()
)


nested_rows = []

validation_rows = []

test_trade_frames = []


# ------------------------------------------------------------
# Validation年が必要なので
#
# Train:
# 最低3年以上
#
# Validation:
# Testの前年
#
# Test:
# その翌年
# ------------------------------------------------------------

for test_year in (
    all_years
):

    validation_year = (
        test_year
        - 1
    )

    train_years = [
        y
        for y in all_years
        if y
        <
        validation_year
    ]

    if (
        len(
            train_years
        )
        <
        MIN_TRAIN_YEARS
    ):

        continue

    train = (
        data.loc[
            data.index.year
            <
            validation_year
        ]
        .copy()
    )

    validation = (
        data.loc[
            data.index.year
            ==
            validation_year
        ]
        .copy()
    )

    test = (
        data.loc[
            data.index.year
            ==
            test_year
        ]
        .copy()
    )

    if (
        len(train)
        <
        5000
        or
        len(validation)
        <
        100
        or
        len(test)
        <
        100
    ):

        continue

    print()
    print(
        "===================================="
    )

    print(
        f"TEST YEAR {test_year}"
    )

    print(
        "===================================="
    )

    print(
        "Train:",
        train.index.year.min(),
        "→",
        train.index.year.max()
    )

    print(
        "Validation:",
        validation_year
    )

    print(
        "Test:",
        test_year
    )

    # ========================================================
    # 13. Train -> Validation
    # ========================================================

    validation_model = (
        build_model()
    )

    validation_model.fit(
        train[
            FEATURES
        ],
        train[
            "target"
        ],
    )

    validation_predictions = (
        predict_frame(
            validation_model,
            validation
        )
    )

    (
        chosen_threshold,
        threshold_table,
    ) = choose_threshold(
        validation_predictions
    )

    if (
        chosen_threshold
        is None
    ):

        print(
            "Threshold選択不可"
        )

        continue

    threshold_table[
        "validation_year"
    ] = (
        validation_year
    )

    threshold_table[
        "test_year"
    ] = (
        test_year
    )

    validation_rows.append(
        threshold_table
    )

    print(
        "選択Threshold:",
        chosen_threshold
    )

    # ========================================================
    # 14. Train + Validationで再学習
    #
    # Thresholdはここでは変えない
    # ========================================================

    final_train = (
        data.loc[
            data.index.year
            <
            test_year
        ]
        .copy()
    )

    final_model = (
        build_model()
    )

    final_model.fit(
        final_train[
            FEATURES
        ],
        final_train[
            "target"
        ],
    )

    # ========================================================
    # 15. Test
    # ========================================================

    test_probabilities = (
        final_model.predict_proba(
            test[
                FEATURES
            ]
        )[:, 1]
    )

    test_auc = (
        roc_auc_score(
            test[
                "target"
            ],
            test_probabilities
        )
    )

    test_predictions = (
        predict_frame(
            final_model,
            test
        )
    )

    selected_test = (
        test_predictions.loc[
            test_predictions[
                "confidence"
            ]
            >=
            chosen_threshold
        ]
    )

    selected_test = (
        non_overlapping(
            selected_test
        )
    )

    net_returns = (
        selected_test[
            "gross_return"
        ]
        -
        TRADING_COST
    )

    test_stats = (
        calculate_stats(
            net_returns
        )
    )

    selected_test[
        "test_year"
    ] = (
        test_year
    )

    selected_test[
        "chosen_threshold"
    ] = (
        chosen_threshold
    )

    test_trade_frames.append(
        selected_test
    )

    nested_rows.append(
        {
            "test_year":
                test_year,

            "validation_year":
                validation_year,

            "chosen_threshold":
                chosen_threshold,

            "test_auc":
                test_auc,

            "trades":
                test_stats[
                    "trades"
                ],

            "accuracy":
                test_stats[
                    "accuracy"
                ],

            "avg_return":
                test_stats[
                    "avg_return"
                ],

            "pf":
                test_stats[
                    "pf"
                ],

            "max_dd":
                test_stats[
                    "max_dd"
                ],

            "growth":
                test_stats[
                    "growth"
                ],
        }
    )

    print(
        "Test AUC:",
        round(
            test_auc,
            4
        )
    )

    print(
        "Trades:",
        test_stats[
            "trades"
        ]
    )

    print(
        "Accuracy:",
        round(
            test_stats[
                "accuracy"
            ]
            *
            100,
            2
        ),
        "%"
    )

    print(
        "Avg Return:",
        round(
            test_stats[
                "avg_return"
            ]
            *
            100,
            5
        ),
        "%"
    )

    print(
        "PF:",
        round(
            test_stats[
                "pf"
            ],
            3
        )
    )


# ============================================================
# 16. Nested結果
# ============================================================

nested_results = pd.DataFrame(
    nested_rows
)


print()
print(
    "===================================="
)

print(
    "NESTED WALK-FORWARD 最終結果"
)

print(
    "===================================="
)


nested_show = (
    nested_results
    .copy()
)


for col in [
    "chosen_threshold",
    "accuracy",
    "avg_return",
    "max_dd",
    "growth",
]:

    nested_show[
        col
    ] *= 100


print(
    nested_show.to_string(
        index=False
    )
)


# ============================================================
# 17. 安定性
# ============================================================

print()
print(
    "===================================="
)

print(
    "安定性"
)

print(
    "===================================="
)


if (
    not nested_results.empty
):

    positive_years = (
        nested_results[
            "avg_return"
        ]
        >
        0
    ).sum()

    pf_positive_years = (
        nested_results[
            "pf"
        ]
        >
        1
    ).sum()

    print(
        "評価年:",
        len(
            nested_results
        )
    )

    print(
        "平均Return > 0:",
        positive_years,
        "/",
        len(
            nested_results
        )
    )

    print(
        "PF > 1:",
        pf_positive_years,
        "/",
        len(
            nested_results
        )
    )

    print(
        "平均PF:",
        nested_results[
            "pf"
        ].mean()
    )

    print(
        "中央値PF:",
        nested_results[
            "pf"
        ].median()
    )

    print(
        "平均Return:",
        nested_results[
            "avg_return"
        ].mean()
        *
        100,
        "%"
    )


# ============================================================
# 18. Threshold選択頻度
# ============================================================

print()
print(
    "===================================="
)

print(
    "Threshold選択頻度"
)

print(
    "===================================="
)


if (
    not nested_results.empty
):

    print(
        nested_results[
            "chosen_threshold"
        ]
        .value_counts()
        .sort_index()
    )


# ============================================================
# 19. 全Test取引を結合
# ============================================================

if (
    test_trade_frames
):

    all_test_trades = (
        pd.concat(
            test_trade_frames
        )
        .sort_index()
    )

    all_net_return = (
        all_test_trades[
            "gross_return"
        ]
        -
        TRADING_COST
    )

    overall_stats = (
        calculate_stats(
            all_net_return
        )
    )

    print()
    print(
        "===================================="
    )

    print(
        "全Nested Test統合"
    )

    print(
        "===================================="
    )

    print(
        "Trades:",
        overall_stats[
            "trades"
        ]
    )

    print(
        "Accuracy:",
        overall_stats[
            "accuracy"
        ]
        *
        100,
        "%"
    )

    print(
        "Avg Return:",
        overall_stats[
            "avg_return"
        ]
        *
        100,
        "%"
    )

    print(
        "PF:",
        overall_stats[
            "pf"
        ]
    )

    print(
        "Max DD:",
        overall_stats[
            "max_dd"
        ]
        *
        100,
        "%"
    )

    print(
        "Growth:",
        overall_stats[
            "growth"
        ]
        *
        100,
        "%"
    )


# ============================================================
# 20. Threshold選択過程
# ============================================================

if (
    validation_rows
):

    validation_results = (
        pd.concat(
            validation_rows,
            ignore_index=True
        )
    )

    print()
    print(
        "===================================="
    )

    print(
        "Validation Threshold探索"
    )

    print(
        "===================================="
    )

    validation_show = (
        validation_results
        .copy()
    )

    validation_show[
        "threshold"
    ] *= 100

    validation_show[
        "accuracy"
    ] *= 100

    validation_show[
        "avg_return"
    ] *= 100

    print(
        validation_show.to_string(
            index=False
        )
    )


# ============================================================
# 21. 年別PFグラフ
# ============================================================

if (
    not nested_results.empty
):

    plt.figure(
        figsize=(
            9,
            5
        )
    )

    plt.plot(
        nested_results[
            "test_year"
        ],
        nested_results[
            "pf"
        ],
        marker="o"
    )

    plt.axhline(
        1,
        linewidth=1
    )

    plt.xlabel(
        "Test Year"
    )

    plt.ylabel(
        "Profit Factor"
    )

    plt.title(
        "Nested Walk-Forward PF"
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 22. Threshold推移
# ============================================================

if (
    not nested_results.empty
):

    plt.figure(
        figsize=(
            9,
            5
        )
    )

    plt.plot(
        nested_results[
            "test_year"
        ],
        nested_results[
            "chosen_threshold"
        ]
        *
        100,
        marker="o"
    )

    plt.xlabel(
        "Test Year"
    )

    plt.ylabel(
        "Chosen Confidence Threshold (%)"
    )

    plt.title(
        "Validation-selected Threshold"
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# 23. 保存
# ============================================================

OUTPUT_DIR = (
    Path.cwd()
    /
    "15m_nested_walk_forward"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)


nested_results.to_csv(
    OUTPUT_DIR
    /
    "nested_results.csv",
    index=False
)


if (
    test_trade_frames
):

    all_test_trades.to_csv(
        OUTPUT_DIR
        /
        "nested_test_trades.csv"
    )


if (
    validation_rows
):

    validation_results.to_csv(
        OUTPUT_DIR
        /
        "validation_threshold_search.csv",
        index=False
    )


print()
print(
    "===================================="
)

print(
    "検証完了"
)

print(
    "===================================="
)

print(
    OUTPUT_DIR.resolve()
)

print()
print(
    "最重要チェック:"
)

print(
    "1. Test年でPF>1が何年あるか"
)

print(
    "2. 選択Thresholdが毎年極端に変わらないか"
)

print(
    "3. 全Nested Test統合PF"
)

print(
    "4. 全Nested Test平均Return"
)

print(
    "5. 直近2025/2026でも維持しているか"
)